# Defects and Materials Modeling

This notebook is a step-by-step tutorial for building, relaxing, and analyzing simple defects in atomistic materials models. The workflow connects three practical tools:

- **LAMMPS** for building and relaxing simple atomistic structures.
- **Atomsk** for thinking about structure construction and file conversion.
- **OVITO** for visualization and structural interpretation.

The examples use a small Lennard-Jones FCC crystal so the LAMMPS scripts are self-contained and do not require a separate potential file. The same workflow transfers to real materials when you replace the model potential with an appropriate validated force field, such as an EAM potential for a metal.

## Learning outcomes

By the end of this notebook, you should be able to:

1. Explain the difference between a perfect crystal, a point defect, and a surface model.
2. Generate LAMMPS input scripts for a perfect FCC crystal, a vacancy, and a slab.
3. Compute vacancy formation energy from relaxed total energies.
4. Estimate surface energy from a slab calculation.
5. Create a clear OVITO visualization checklist for checking whether a model is physically reasonable.
6. Identify what must be validated before trusting a defect simulation.


## 0. How to Use This Notebook

This notebook is designed for both classroom teaching and independent learning.

- Run the Python cells first. They create folders, write LAMMPS scripts, and provide analysis helpers.
- If LAMMPS is installed, use the optional run cells to execute the scripts.
- If LAMMPS is not installed, read the generated input scripts and use the provided example energies to practice the calculations.
- Use OVITO after the LAMMPS run, or use the visualization checklist to discuss what you would inspect.

The key habit is not simply "run a simulation." The key habit is:

**build -> relax -> measure -> visualize -> question -> validate**


In [ ]:
from pathlib import Path
import shutil
import subprocess
import textwrap
import re

ROOT = Path.cwd()
DIRS = {
    "lammps": ROOT / "lammps",
    "atomsk": ROOT / "atomsk",
    "ovito": ROOT / "ovito",
    "data": ROOT / "data",
    "figures": ROOT / "figures",
    "logs": ROOT / "logs",
}

for path in DIRS.values():
    path.mkdir(exist_ok=True)

def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).strip() + "\n", encoding="utf-8")
    print(f"Wrote {path}")

def show_file(path, max_lines=80):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    for i, line in enumerate(lines[:max_lines], start=1):
        print(f"{i:03d}: {line}")
    if len(lines) > max_lines:
        print(f"... {len(lines) - max_lines} more lines")

def find_lammps_command():
    candidates = ["lmp", "lmp_serial", "lammps", "lmp_mpi"]
    for name in candidates:
        if shutil.which(name):
            return name
    return None

print("Workspace folders are ready:")
for name, path in DIRS.items():
    print(f"  {name:7s} -> {path}")
print("LAMMPS command found:", find_lammps_command() or "not found")


## 1. Start with the Scientific Question

A defect simulation should begin with a question that can be answered by a specific model.

In this tutorial we will use three introductory questions:

1. **Perfect crystal reference:** What is the relaxed energy per atom of the perfect FCC crystal?
2. **Vacancy defect:** How much energy does it cost to remove one atom from the crystal?
3. **Surface model:** How much excess energy is associated with creating two free surfaces?

These are simple questions, but they teach the structure of many larger materials modeling workflows.

### Checkpoint

Before you simulate, write down:

- What structure am I modeling?
- What quantity will I calculate?
- What reference state do I need?
- What assumptions are built into the potential, boundary conditions, and cell size?


In [ ]:
workflow = """
Defect modeling workflow

1. Define the scientific question.
2. Build a perfect reference structure.
3. Relax the perfect structure and record energy and atom count.
4. Build a modified structure, such as a vacancy or slab.
5. Relax the modified structure with the same model settings.
6. Compute a derived quantity, such as defect formation energy or surface energy.
7. Visualize the initial and relaxed structures.
8. Check whether the result is physically reasonable.
9. Repeat with a larger cell or different settings to test sensitivity.
"""

write_text(DIRS["data"] / "defect_modeling_workflow.txt", workflow)
print(workflow)


## 2. Build a Perfect FCC Reference Crystal in LAMMPS

The perfect crystal is the reference state. We need it because a defect energy is usually not just the total energy of the defective structure. It is the energy of the defective structure compared with the same number of atoms in an equivalent perfect crystal.

The script below creates a small FCC Lennard-Jones crystal directly inside LAMMPS, minimizes it, and prints two values we need later:

- number of atoms
- relaxed potential energy

For a real metal, you would usually replace the Lennard-Jones settings with an EAM potential and a realistic lattice constant.


In [ ]:
bulk_script = r"""
# Perfect FCC Lennard-Jones crystal: relaxation reference
units           lj
atom_style      atomic
boundary        p p p

variable        nx equal 6
variable        ny equal 6
variable        nz equal 6

lattice         fcc 0.8442
region          box block 0 ${nx} 0 ${ny} 0 ${nz} units lattice
create_box      1 box
create_atoms    1 box

mass            1 1.0
pair_style      lj/cut 2.5
pair_coeff      1 1 1.0 1.0 2.5

neighbor        0.3 bin
neigh_modify    delay 0 every 1 check yes

thermo          50
thermo_style    custom step atoms pe etotal press lx ly lz

min_style       cg
minimize        1.0e-10 1.0e-12 1000 10000

variable        nall equal count(all)
variable        epe equal pe
print           "RESULT bulk_atoms ${nall}"
print           "RESULT bulk_pe ${epe}"

write_data      data/lj_bulk_relaxed.data
write_dump      all atom data/lj_bulk_relaxed.dump
"""

write_text(DIRS["lammps"] / "in.01_bulk_relax.lmp", bulk_script)
show_file(DIRS["lammps"] / "in.01_bulk_relax.lmp", max_lines=70)


## 3. Create and Relax a Vacancy

A vacancy is created by removing one atom from an otherwise perfect crystal. The important modeling rule is that the perfect and vacancy calculations must use the same potential, cutoff, minimization settings, and comparable cell size.

The script below creates the same FCC crystal, deletes one atom near the origin, relaxes the structure, and prints the atom count and potential energy.

### Checkpoint

After the run, confirm that the vacancy cell has exactly one fewer atom than the perfect cell. If it does not, the formation energy calculation is not valid.


In [ ]:
vacancy_script = r"""
# FCC Lennard-Jones crystal with one vacancy
units           lj
atom_style      atomic
boundary        p p p

variable        nx equal 6
variable        ny equal 6
variable        nz equal 6

lattice         fcc 0.8442
region          box block 0 ${nx} 0 ${ny} 0 ${nz} units lattice
create_box      1 box
create_atoms    1 box

# Delete one atom at the origin. The small spherical region should contain one FCC site.
region          vacancy_site sphere 0.0 0.0 0.0 0.20 units lattice
group           vacancy_atom region vacancy_site
delete_atoms    group vacancy_atom

mass            1 1.0
pair_style      lj/cut 2.5
pair_coeff      1 1 1.0 1.0 2.5

neighbor        0.3 bin
neigh_modify    delay 0 every 1 check yes

thermo          50
thermo_style    custom step atoms pe etotal press lx ly lz

min_style       cg
minimize        1.0e-10 1.0e-12 1000 10000

variable        nall equal count(all)
variable        epe equal pe
print           "RESULT vacancy_atoms ${nall}"
print           "RESULT vacancy_pe ${epe}"

write_data      data/lj_vacancy_relaxed.data
write_dump      all atom data/lj_vacancy_relaxed.dump
"""

write_text(DIRS["lammps"] / "in.02_vacancy_relax.lmp", vacancy_script)
show_file(DIRS["lammps"] / "in.02_vacancy_relax.lmp", max_lines=85)


## 4. Optional: Run LAMMPS from the Notebook

If LAMMPS is installed and available as `lmp`, `lmp_serial`, `lammps`, or `lmp_mpi`, this cell will run the two relaxation scripts.

If the cell reports that LAMMPS is not found, that is okay for a classroom walkthrough. You can still inspect the scripts and use the example results in the next section.


In [ ]:
def run_lammps(input_script, log_name):
    command = find_lammps_command()
    if command is None:
        print("LAMMPS was not found on PATH. Skipping run.")
        return None

    input_script = Path(input_script)
    log_path = DIRS["logs"] / log_name
    result = subprocess.run(
        [command, "-in", str(input_script), "-log", str(log_path)],
        cwd=ROOT,
        text=True,
        capture_output=True,
    )
    print(result.stdout[-1000:])
    if result.returncode != 0:
        print(result.stderr[-1000:])
        raise RuntimeError(f"LAMMPS failed for {input_script}")
    print(f"Wrote log: {log_path}")
    return log_path

bulk_log = run_lammps(DIRS["lammps"] / "in.01_bulk_relax.lmp", "bulk_relax.log")
vacancy_log = run_lammps(DIRS["lammps"] / "in.02_vacancy_relax.lmp", "vacancy_relax.log")


## 5. Extract Results from LAMMPS Logs

The LAMMPS scripts print lines beginning with `RESULT`. This makes the logs easier to parse and reduces the chance that students copy the wrong number from the thermo table.

If you did not run LAMMPS, the code below uses example values so you can practice the formation-energy calculation.


In [ ]:
def parse_result_lines(log_path):
    values = {}
    if log_path is None or not Path(log_path).exists():
        return values
    pattern = re.compile(r"RESULT\s+(\S+)\s+([-+0-9.eE]+)")
    for line in Path(log_path).read_text(errors="ignore").splitlines():
        match = pattern.search(line)
        if match:
            key, value = match.groups()
            values[key] = float(value)
    return values

results = {}
results.update(parse_result_lines(DIRS["logs"] / "bulk_relax.log"))
results.update(parse_result_lines(DIRS["logs"] / "vacancy_relax.log"))

if not results:
    print("Using example values because no LAMMPS logs were found.")
    results = {
        "bulk_atoms": 864,
        "bulk_pe": -5832.74,
        "vacancy_atoms": 863,
        "vacancy_pe": -5823.18,
    }

for key, value in results.items():
    print(f"{key:15s} = {value}")


## 6. Calculate Vacancy Formation Energy

For a single vacancy in a cell with `N` atoms before deletion, the vacancy formation energy is

$$E_f^v = E_{N-1}^{vac} - \frac{N-1}{N}E_N^{bulk}$$

where:

- $E_N^{bulk}$ is the relaxed total energy of the perfect crystal.
- $E_{N-1}^{vac}$ is the relaxed total energy of the vacancy crystal.
- $N$ is the number of atoms in the perfect cell.

The factor $(N-1)/N$ compares the vacancy cell with the same number of atoms in the perfect reference state.


In [ ]:
def vacancy_formation_energy(e_bulk, n_bulk, e_vac, n_vac):
    expected = int(n_bulk) - 1
    if int(n_vac) != expected:
        raise ValueError(
            f"Expected vacancy cell to contain {expected} atoms, "
            f"but got {n_vac}. Check the delete_atoms command."
        )
    return e_vac - (n_vac / n_bulk) * e_bulk

ef_vac = vacancy_formation_energy(
    e_bulk=results["bulk_pe"],
    n_bulk=results["bulk_atoms"],
    e_vac=results["vacancy_pe"],
    n_vac=results["vacancy_atoms"],
)

print(f"Vacancy formation energy = {ef_vac:.4f} LJ energy units")
print("For a real material, report the value in eV after using a validated potential.")


## 7. Build a Surface Slab Model

Surfaces are another common defect type. A slab model creates free surfaces by making one direction non-periodic.

The surface energy is often estimated as

$$\gamma = \frac{E_{slab} - N_{slab}E_{bulk/atom}}{2A}$$

where:

- $E_{slab}$ is the relaxed slab energy.
- $N_{slab}$ is the number of slab atoms.
- $E_{bulk/atom}$ is the relaxed bulk energy per atom.
- $A$ is the area of one surface.
- The factor of 2 appears because the slab has two free surfaces.

This is a teaching example. Production surface-energy calculations need careful convergence with slab thickness, vacuum spacing, surface orientation, and relaxation settings.


In [ ]:
slab_script = r"""
# FCC Lennard-Jones slab with two free surfaces normal to z
units           lj
atom_style      atomic
boundary        p p f

variable        nx equal 8
variable        ny equal 8
variable        nz equal 8

lattice         fcc 0.8442
region          box block 0 ${nx} 0 ${ny} -2 12 units lattice
create_box      1 box
region          slab block 0 ${nx} 0 ${ny} 0 ${nz} units lattice
create_atoms    1 region slab

mass            1 1.0
pair_style      lj/cut 2.5
pair_coeff      1 1 1.0 1.0 2.5

neighbor        0.3 bin
neigh_modify    delay 0 every 1 check yes

thermo          50
thermo_style    custom step atoms pe etotal press lx ly lz

min_style       cg
minimize        1.0e-10 1.0e-12 1000 10000

variable        nall equal count(all)
variable        epe equal pe
variable        area equal lx*ly
print           "RESULT slab_atoms ${nall}"
print           "RESULT slab_pe ${epe}"
print           "RESULT slab_area ${area}"

write_data      data/lj_slab_relaxed.data
write_dump      all atom data/lj_slab_relaxed.dump
"""

write_text(DIRS["lammps"] / "in.03_slab_relax.lmp", slab_script)
show_file(DIRS["lammps"] / "in.03_slab_relax.lmp", max_lines=90)


In [ ]:
slab_log = run_lammps(DIRS["lammps"] / "in.03_slab_relax.lmp", "slab_relax.log")

slab_results = parse_result_lines(DIRS["logs"] / "slab_relax.log")
if not slab_results:
    print("Using example slab values because no LAMMPS slab log was found.")
    slab_results = {
        "slab_atoms": 2048,
        "slab_pe": -13710.0,
        "slab_area": 180.0,
    }

for key, value in slab_results.items():
    print(f"{key:15s} = {value}")


In [ ]:
def surface_energy(e_slab, n_slab, e_bulk, n_bulk, area):
    e_bulk_per_atom = e_bulk / n_bulk
    excess_energy = e_slab - n_slab * e_bulk_per_atom
    return excess_energy / (2.0 * area)

gamma = surface_energy(
    e_slab=slab_results["slab_pe"],
    n_slab=slab_results["slab_atoms"],
    e_bulk=results["bulk_pe"],
    n_bulk=results["bulk_atoms"],
    area=slab_results["slab_area"],
)

print(f"Estimated surface energy = {gamma:.5f} LJ energy / LJ area")
print("Check whether the sign and magnitude are reasonable before interpreting the result.")


## 8. Where Atomsk Fits

LAMMPS can create simple structures directly, which is useful for teaching. Atomsk becomes helpful when you need to build, orient, convert, or modify more realistic atomistic structures.

The commands below are written to a text file as a planning aid. They are not executed by Python. If Atomsk is installed, you can copy them into a terminal and adapt the lattice constant, element, orientation, and output format.


In [ ]:
atomsk_notes = r"""
# Example Atomsk command patterns for materials defect modeling
# These are command examples; check Atomsk documentation before production use.

# 1. Build an FCC aluminum supercell and export LAMMPS data format.
atomsk --create fcc 4.05 Al orient [100] [010] [001] -duplicate 8 8 8 Al_fcc_8x8x8.lmp

# 2. Create a vacancy by removing one atom near the cell center.
# Exact selection syntax may depend on Atomsk version and structure coordinates.
atomsk Al_fcc_8x8x8.lmp -select in sphere 16.2 16.2 16.2 1.0 -remove-atoms select Al_fcc_vacancy.lmp

# 3. Create a slab by adding vacuum along z.
atomsk Al_fcc_8x8x8.lmp -cell add 0 0 30 Al_fcc_slab.lmp

# 4. Convert a LAMMPS data file to XYZ for quick visualization.
atomsk Al_fcc_vacancy.lmp Al_fcc_vacancy.xyz
"""

write_text(DIRS["atomsk"] / "atomsk_defect_command_patterns.txt", atomsk_notes)
print(atomsk_notes)


## 9. OVITO Visualization Checklist

Numerical values are not enough. Every defect model should be visualized.

Open the generated dump or data files in OVITO if you ran LAMMPS:

- `data/lj_bulk_relaxed.dump`
- `data/lj_vacancy_relaxed.dump`
- `data/lj_slab_relaxed.dump`

Use this checklist while viewing each structure:

1. Does the atom count match the intended model?
2. Does the vacancy appear where expected?
3. Are atoms overlapping or flying apart?
4. Do periodic boundaries make sense for the question?
5. For the slab, are there two free surfaces and enough vacuum?
6. Does relaxation change the local environment around the defect?
7. Would a larger cell change the answer?


In [ ]:
ovito_checklist = """
OVITO checklist for defect and materials modeling

Perfect crystal:
- Confirm periodic FCC ordering.
- Use Common Neighbor Analysis if available.
- Check for unexpected surfaces or voids.

Vacancy model:
- Confirm exactly one missing atom.
- Inspect nearest-neighbor relaxation around the vacancy.
- Compare before and after minimization if both files are available.

Surface slab:
- Confirm the non-periodic direction is the intended surface normal.
- Confirm vacuum exists above and below the slab.
- Color atoms by z-position or coordination to identify surface atoms.

General validation:
- Compare atom count, energy, and box dimensions against the LAMMPS log.
- Save one image for the notebook or lab report.
"""

write_text(DIRS["ovito"] / "visualization_checklist.md", ovito_checklist)
print(ovito_checklist)


## 10. Validation and Teaching Discussion

Use the questions below for class discussion, homework, or a lab report.

### Model validation

- Why should the perfect and vacancy calculations use the same cell size and potential settings?
- What would happen if the vacancy cell accidentally removed two atoms?
- Why do surfaces require non-periodic boundary conditions in one direction?
- Why should a surface-energy result be checked against slab thickness and vacuum spacing?

### AI-assisted workflow validation

If an AI assistant helps draft a LAMMPS or Atomsk command, ask:

- Does the command match the software documentation?
- Are the units, boundary conditions, and potential style physically appropriate?
- Does the script print the quantities needed for analysis?
- Can another student reproduce the result from the files and instructions?


## 11. Student Exercises

Complete these exercises in order.

1. Run the perfect-crystal script and record `bulk_atoms` and `bulk_pe`.
2. Run the vacancy script and record `vacancy_atoms` and `vacancy_pe`.
3. Compute the vacancy formation energy using the provided function.
4. Open the bulk and vacancy dump files in OVITO and describe what changed near the vacancy.
5. Run the slab script and compute the surface energy.
6. Change the cell size from `6 x 6 x 6` to `8 x 8 x 8` for the vacancy calculation. Does the vacancy formation energy change?
7. Write a short paragraph explaining which result you trust more and why.

## Instructor Notes

A good class implementation is to have students work in pairs:

- One student reads the LAMMPS input script line by line.
- One student tracks the physical meaning of each modeling choice.
- Both students compare their numerical results with their OVITO observations.

The main teaching goal is not the exact Lennard-Jones value. The main goal is a reproducible workflow for defect modeling that students can later transfer to real materials and validated interatomic potentials.
